In [1]:
! python -m pip install librosa soundfile numpy pandas tqdm praat-parselmouth

  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.3/1.0 MB ? eta -:--:--
   -------------------- ------------------- 0.5/1.0 MB 2.0 MB/s eta 0:00:01
   ---------------------------------------- 1.0/1.0 MB 1.5 MB/s  0:00:00
Using cached tqdm-4.67.3-py3-none-any.whl (78 kB)
   ---------------------------------------- 0.0/9.0 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.0 MB ? eta -:--:--
   --- ------------------------------------ 0.8/9.0 MB 2.1 MB/s eta 0:00:04
   ---- ----------------------------------- 1.0/9.0 MB 1.8 MB/s eta 0:00:05
   ----- ---------------------------------- 1.3/9.0 MB 1.7 MB/s eta 0:00:05
   ------- -------------------------------- 1.6/9.0 MB 1.6 MB/s eta 0:00:05
   --------- ------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import librosa
import numpy as np
import pandas as pd
import parselmouth
from parselmouth.praat import call
import os
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

SAMPLE_RATE = 16000
langs = ['malayalam', 'tamil', 'hindi', 'english', 'kannada']

## Load Cleaned metadata

In [3]:
df = pd.read_csv('C:\\Users\\Shipra\\OneDrive\\Documents\\GitHub\\predictive-project-grp-2\\predictive-project-grp-2\\data\\metadata\\metadata_clean.csv')
print(f"Total clips to process: {len(df)}")
print(df.groupby('language').size())

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Shipra\\OneDrive\\Documents\\GitHub\\predictive-project-grp-2\\predictive-project-grp-2\\data\\metadata\\metadata_clean.csv'

## Extract MFCC(39 COEFFICIENTS)

In [ ]:
def extract_mfcc(y, sr, n_mfcc=13):
    """
    Returns 39 features:
    - 13 base MFCC mean + 13 base MFCC std
    - 13 delta mean + 13 delta std  
    - 13 delta-delta mean + 13 delta-delta std
    Total = 78 values (mean+std for each of 39 coefficients)
    """
    # Base MFCC
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc,
                                  n_fft=512, hop_length=160, win_length=400)
    # Delta (velocity)
    mfcc_delta = librosa.feature.delta(mfcc)
    # Delta-delta (acceleration)
    mfcc_delta2 = librosa.feature.delta(mfcc, order=2)

    features = {}

    for i in range(n_mfcc):
        features[f'mfcc_{i+1}_mean']        = np.mean(mfcc[i])
        features[f'mfcc_{i+1}_std']         = np.std(mfcc[i])
        features[f'mfcc_d_{i+1}_mean']      = np.mean(mfcc_delta[i])
        features[f'mfcc_d_{i+1}_std']       = np.std(mfcc_delta[i])
        features[f'mfcc_d2_{i+1}_mean']     = np.mean(mfcc_delta2[i])
        features[f'mfcc_d2_{i+1}_std']      = np.std(mfcc_delta2[i])

    return features

## Extract Pitch(F0)

In [ ]:
def extract_pitch(y, sr):
    """
    Extracts fundamental frequency (F0) using parselmouth/Praat.
    Returns mean, std, min, max, and fraction of voiced frames.
    """
    # Write to temp wav for parselmouth
    import soundfile as sf
    import tempfile

    with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as tmp:
        tmp_path = tmp.name
    sf.write(tmp_path, y, sr)

    try:
        sound = parselmouth.Sound(tmp_path)
        pitch = call(sound, "To Pitch", 0.0, 75, 600)  # 75–600 Hz range
        pitch_values = pitch.selected_array['frequency']
        voiced = pitch_values[pitch_values > 0]  # remove unvoiced (0 Hz) frames

        if len(voiced) == 0:
            features = {
                'pitch_mean': 0.0, 'pitch_std': 0.0,
                'pitch_min': 0.0,  'pitch_max': 0.0,
                'voiced_fraction': 0.0
            }
        else:
            features = {
                'pitch_mean':       float(np.mean(voiced)),
                'pitch_std':        float(np.std(voiced)),
                'pitch_min':        float(np.min(voiced)),
                'pitch_max':        float(np.max(voiced)),
                'voiced_fraction':  float(len(voiced) / len(pitch_values))
            }
    except:
        features = {
            'pitch_mean': 0.0, 'pitch_std': 0.0,
            'pitch_min': 0.0,  'pitch_max': 0.0,
            'voiced_fraction': 0.0
        }
    finally:
        os.remove(tmp_path)

    return features

## Extract Energy(RMS)

In [ ]:
def extract_energy(y, sr):
    """
    RMS energy — captures loudness and rhythmic stress patterns.
    Returns mean, std, max, and dynamic range.
    """
    rms = librosa.feature.rms(y=y, frame_length=400, hop_length=160)[0]

    features = {
        'energy_mean':          float(np.mean(rms)),
        'energy_std':           float(np.std(rms)),
        'energy_max':           float(np.max(rms)),
        'energy_dynamic_range': float(np.max(rms) - np.min(rms))
    }
    return features

## Extract Formants F1,F2,F3

In [ ]:
def extract_formants(y, sr):
    """
    Formant frequencies F1, F2, F3 using Praat's Burg algorithm.
    These capture vowel space — critical for Dravidian vs Indo-Aryan distinction.
    """
    import soundfile as sf
    import tempfile

    with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as tmp:
        tmp_path = tmp.name
    sf.write(tmp_path, y, sr)

    try:
        sound = parselmouth.Sound(tmp_path)
        formants = call(sound, "To Formant (burg)", 0.0, 5, 5500, 0.025, 50)

        n_frames = call(formants, "Get number of frames")

        f1_vals, f2_vals, f3_vals = [], [], []

        for frame in range(1, n_frames + 1):
            f1 = call(formants, "Get value at time", 1,
                      call(formants, "Get time from frame number", frame),
                      'Hertz', 'Linear')
            f2 = call(formants, "Get value at time", 2,
                      call(formants, "Get time from frame number", frame),
                      'Hertz', 'Linear')
            f3 = call(formants, "Get value at time", 3,
                      call(formants, "Get time from frame number", frame),
                      'Hertz', 'Linear')

            if f1 and not np.isnan(f1): f1_vals.append(f1)
            if f2 and not np.isnan(f2): f2_vals.append(f2)
            if f3 and not np.isnan(f3): f3_vals.append(f3)

        features = {
            'F1_mean': float(np.mean(f1_vals)) if f1_vals else 0.0,
            'F1_std':  float(np.std(f1_vals))  if f1_vals else 0.0,
            'F2_mean': float(np.mean(f2_vals)) if f2_vals else 0.0,
            'F2_std':  float(np.std(f2_vals))  if f2_vals else 0.0,
            'F3_mean': float(np.mean(f3_vals)) if f3_vals else 0.0,
            'F3_std':  float(np.std(f3_vals))  if f3_vals else 0.0,
        }

    except:
        features = {
            'F1_mean': 0.0, 'F1_std': 0.0,
            'F2_mean': 0.0, 'F2_std': 0.0,
            'F3_mean': 0.0, 'F3_std': 0.0,
        }
    finally:
        os.remove(tmp_path)

    return features

## Extract Spectral Features

In [ ]:
def extract_spectral(y, sr):
    """
    Spectral centroid, bandwidth, rolloff, zero crossing rate.
    These capture the brightness and texture of speech.
    """
    centroid  = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
    rolloff   = librosa.feature.spectral_rolloff(y=y, sr=sr, roll_percent=0.85)[0]
    zcr       = librosa.feature.zero_crossing_rate(y)[0]

    features = {
        'spectral_centroid_mean':  float(np.mean(centroid)),
        'spectral_centroid_std':   float(np.std(centroid)),
        'spectral_bandwidth_mean': float(np.mean(bandwidth)),
        'spectral_bandwidth_std':  float(np.std(bandwidth)),
        'spectral_rolloff_mean':   float(np.mean(rolloff)),
        'spectral_rolloff_std':    float(np.std(rolloff)),
        'zcr_mean':                float(np.mean(zcr)),
        'zcr_std':                 float(np.std(zcr)),
    }
    return features

## Vowel/Consonant ratio

In [ ]:
def extract_phonotactic(y, sr):
    """
    Estimates vowel/consonant ratio using energy and ZCR heuristics.
    - High energy + low ZCR frames → likely vowels (voiced, resonant)
    - Low energy + high ZCR frames → likely consonants (unvoiced, fricatives)
    Dravidian languages (Malayalam, Tamil) tend to have higher vowel ratios.
    """
    frame_len  = 400
    hop_len    = 160

    rms = librosa.feature.rms(y=y, frame_length=frame_len, hop_length=hop_len)[0]
    zcr = librosa.feature.zero_crossing_rate(y, frame_length=frame_len, hop_length=hop_len)[0]

    # Normalize
    rms_norm = rms / (np.max(rms) + 1e-8)
    zcr_norm = zcr / (np.max(zcr) + 1e-8)

    # Vowel-like frames: high energy, low ZCR
    vowel_frames     = np.sum((rms_norm > 0.3) & (zcr_norm < 0.3))
    consonant_frames = np.sum((rms_norm < 0.3) | (zcr_norm > 0.5))
    total_frames     = len(rms)

    vc_ratio = vowel_frames / (consonant_frames + 1e-8)

    features = {
        'vowel_ratio':     float(vowel_frames / total_frames),
        'consonant_ratio': float(consonant_frames / total_frames),
        'vc_ratio':        float(vc_ratio),
    }
    return features

## Main extraction loop- it runs all the functions on every clip

In [ ]:
all_features = []
failed_clips  = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Extracting features"):
    try:
        y, sr = librosa.load(row['file_path'], sr=SAMPLE_RATE, mono=True)

        # Run all feature extractors
        feat = {}
        feat['file_name'] = row['file_name']
        feat['language']  = row['language']
        feat['duration']  = row['duration_sec']

        feat.update(extract_mfcc(y, sr))
        feat.update(extract_energy(y, sr))
        feat.update(extract_spectral(y, sr))
        feat.update(extract_phonotactic(y, sr))

        # Pitch and formants use parselmouth (slightly slower)
        feat.update(extract_pitch(y, sr))
        feat.update(extract_formants(y, sr))

        all_features.append(feat)

    except Exception as e:
        failed_clips.append((row['file_name'], str(e)))

print(f"\nSuccessfully extracted: {len(all_features)} clips")
print(f"Failed: {len(failed_clips)} clips")
if failed_clips:
    for fc in failed_clips[:5]:
        print(f"  {fc}")

Extracting features:   0%|          | 0/1460 [00:00<?, ?it/s]

Extracting features: 100%|██████████| 1460/1460 [00:05<00:00, 278.41it/s]


Successfully extracted: 0 clips
Failed: 1460 clips
  ('f0004_us_f0004_00065.wav', "[Errno 2] No such file or directory: 'data/processed/english\\\\f0004_us_f0004_00065.wav'")
  ('m0005_us_m0005_00127.wav', "[Errno 2] No such file or directory: 'data/processed/english\\\\m0005_us_m0005_00127.wav'")
  ('taf_06958_01627639124.wav', "[Errno 2] No such file or directory: 'data/processed/tamil\\\\taf_06958_01627639124.wav'")
  ('taf_06478_01298281548.wav', "[Errno 2] No such file or directory: 'data/processed/tamil\\\\taf_06478_01298281548.wav'")
  ('f0001_us_f0001_00056.wav', "[Errno 2] No such file or directory: 'data/processed/english\\\\f0001_us_f0001_00056.wav'")


## Save Feature matrix

In [ ]:
df_features = pd.DataFrame(all_features)

os.makedirs('data/features', exist_ok=True)
df_features.to_csv('data/features/features_raw.csv', index=False)

print(f"Feature matrix shape: {df_features.shape}")
print(f"  Rows (clips):    {df_features.shape[0]}")
print(f"  Columns (feats): {df_features.shape[1]}")
print(f"\nFeature groups:")
mfcc_cols     = [c for c in df_features.columns if 'mfcc' in c]
pitch_cols    = [c for c in df_features.columns if 'pitch' in c or 'voiced' in c]
energy_cols   = [c for c in df_features.columns if 'energy' in c]
formant_cols  = [c for c in df_features.columns if c.startswith('F')]
spectral_cols = [c for c in df_features.columns if 'spectral' in c or 'zcr' in c]
phonot_cols   = [c for c in df_features.columns if 'ratio' in c]

print(f"  MFCC features:      {len(mfcc_cols)}")
print(f"  Pitch features:     {len(pitch_cols)}")
print(f"  Energy features:    {len(energy_cols)}")
print(f"  Formant features:   {len(formant_cols)}")
print(f"  Spectral features:  {len(spectral_cols)}")
print(f"  Phonotactic feats:  {len(phonot_cols)}")
print(f"\nSample (first 5 rows, first 8 columns):")
df_features.iloc[:5, :8]

Feature matrix shape: (0, 0)
  Rows (clips):    0
  Columns (feats): 0

Feature groups:
  MFCC features:      0
  Pitch features:     0
  Energy features:    0
  Formant features:   0
  Spectral features:  0
  Phonotactic feats:  0

Sample (first 5 rows, first 8 columns):


""


## Check for nulls and fix them

In [ ]:
null_counts = df_features.isnull().sum()
null_cols   = null_counts[null_counts > 0]

if len(null_cols) > 0:
    print(f"Columns with nulls:\n{null_cols}")
    # Fill nulls with column mean
    feature_cols = df_features.columns.difference(['file_name','language','duration'])
    df_features[feature_cols] = df_features[feature_cols].fillna(
        df_features[feature_cols].mean()
    )
    print("Nulls filled with column mean.")
else:
    print("No nulls found. Feature matrix is clean.")

df_features.to_csv('data/features/features_raw.csv', index=False)
print("Saved: data/features/features_raw.csv")

No nulls found. Feature matrix is clean.
Saved: data/features/features_raw.csv


# feature selection

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import f_classif, SelectKBest
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings('ignore')

LANG_COLORS = {
    'malayalam': '#7F77DD', 'tamil': '#1D9E75',
    'hindi': '#D85A30', 'english': '#378ADD', 'kannada': '#BA7517'
}

In [ ]:
import os

path = r'C:\Users\Shipra\OneDrive\Documents\GitHub\predictive-project-grp-2\data\features\features_raw.csv'

print(f"File exists: {os.path.exists(path)}")
print(f"File size:   {os.path.getsize(path)} bytes")

File exists: True
File size:   2 bytes


In [ ]:
import pandas as pd

df = pd.read_csv(r'C:\Users\Shipra\OneDrive\Documents\GitHub\predictive-project-grp-2\predictive-project-grp-2\data\metadata\metadata_clean.csv')
print(f"Clips in metadata: {len(df)}")
print(df.head(3))
print(df['file_path'].iloc[0])

ParserError: Error tokenizing data. C error: out of memory

In [ ]:
df_2 = pd.read_csv(r'C:\\Users\\Shipra\\OneDrive\\Documents\\GitHub\\predictive-project-grp-2\\data\\features\\features_raw.csv')

meta_cols    = ['file_name', 'language', 'duration']
feature_cols = [c for c in df_2.columns if c not in meta_cols]

X = df_2[feature_cols]
y = df_2['language']

le = LabelEncoder()
y_encoded = le.fit_transform(y)

print(f"Feature matrix: {X.shape[0]} clips × {X.shape[1]} features")
print(f"Classes: {list(le.classes_)}")

EmptyDataError: No columns to parse from file